In [1]:
import pandas as pd
import numpy as np

Sydney dataset

In [ ]:
CSV_PATH = r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/sydney/img_info.csv"

# The CSV has NO header row. Prevent pandas from treating the first data row as column names.
COLS = ["image_id", "lon", "lat", "city", "sat_path", "pano_path"]

df_raw = pd.read_csv(CSV_PATH, header=None, names=COLS)

# ── Basic checks ─────────────────────────────────────────────────────────────
print("Raw shape:", df_raw.shape)
print("Raw dtypes:\n", df_raw.dtypes)
print("Raw NA counts:\n", df_raw.isna().sum())

# empty-string checks (common in CSVs)
empty_counts = (df_raw.astype(str).apply(lambda s: s.str.strip().eq(""))).sum()
print("\nEmpty-string counts:\n", empty_counts)

# ── Cleaning ─────────────────────────────────────────────────────────────────
df = df_raw.copy()

# Trim whitespace in string columns
for c in ["image_id", "city", "sat_path", "pano_path"]:
    df[c] = df[c].astype(str).str.strip()

# Coerce lon/lat to numeric (invalid parses become NaN)
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")

# Convert common missing-string tokens to NA
missing_tokens = {"", "nan", "none", "null"}
for c in ["image_id", "city", "sat_path", "pano_path"]:
    df[c] = df[c].replace({t: pd.NA for t in missing_tokens})

# Drop rows missing critical fields
required = ["image_id", "lon", "lat", "sat_path", "pano_path"]
before = len(df)
df = df.dropna(subset=required)
print(f"\nDropped {before - len(df)} rows with missing required fields")

# Drop duplicate image_id (keep first)
before = len(df)
df = df.drop_duplicates(subset=["image_id"], keep="first")
print(f"Dropped {before - len(df)} duplicate image_id rows")

# Basic geo sanity checks (Sydney bounds are roughly lon~[150, 152], lat~[-34.5, -33])
before = len(df)
df = df[(df["lon"].between(-180, 180)) & (df["lat"].between(-90, 90))]
print(f"Dropped {before - len(df)} rows outside valid geo ranges")

print("\nCleaned shape:", df.shape)
print("Cleaned NA counts:\n", df.isna().sum())

df.head()

In [ ]:
# Downstream cells should use the cleaned `df` created above.
# If you still want a detailed summary, keep it in-notebook:

def report_df(dfx, name="df"):
    print(f"\n=== {name} summary ===")
    print("shape:", dfx.shape)
    print("dtypes:\n", dfx.dtypes)
    na = dfx.isna().sum()
    print("NA counts (non-zero):\n", na[na > 0])
    if {"lon", "lat"}.issubset(dfx.columns):
        print("lon range:", (float(dfx["lon"].min()), float(dfx["lon"].max())))
        print("lat range:", (float(dfx["lat"].min()), float(dfx["lat"].max())))

report_df(df_raw, "df_raw")
report_df(df, "df_clean")

# Optional: write cleaned CSV next to the original
# Requirement: remove pano_path from the exported CSV
CLEAN_OUT = r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/sydney/img_info_clean.csv"
df_export = df.drop(columns=["pano_path"], errors="ignore")
df_export.to_csv(CLEAN_OUT, index=False)
print("\nWrote cleaned CSV (without pano_path) to:", CLEAN_OUT)

# preview exported columns
(df_export.sample(5, random_state=0) if len(df_export) >= 5 else df_export)

Tokoyo dataset Loading

In [2]:
# ── Tokyo: download first 100 images & rewrite CSV with local paths ───────────

import os
import re
import time
import urllib.parse
import urllib.request
from pathlib import Path

TOKYO_CSV = Path(r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/tokyo_clean.csv")
TOKYO_IMG_DIR = Path(r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/tokoyo_images")
TOKYO_OUT_CSV = Path(r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/tokyo_first100_local.csv")

N_DOWNLOAD = 200
TIMEOUT_S = 20
SLEEP_S = 0.2

TOKYO_IMG_DIR.mkdir(parents=True, exist_ok=True)

# Load CSV (it already has headers)
df_tokyo = pd.read_csv(TOKYO_CSV)
print("tokyo_clean.csv shape:", df_tokyo.shape)
print("columns:", list(df_tokyo.columns))

URL_COL = "photo/video_page_url"
if URL_COL not in df_tokyo.columns:
    raise KeyError(f"Expected column '{URL_COL}' not found. Columns are: {list(df_tokyo.columns)}")

sample = df_tokyo.iloc[:N_DOWNLOAD].copy()
print("Processing rows:", len(sample))

UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"


def _safe_filename(s: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", s)
    return s[:180] if len(s) > 180 else s


def _fetch_html(url: str) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=TIMEOUT_S) as resp:
        # Flickr pages are html; decode safely
        data = resp.read()
    return data.decode("utf-8", errors="replace")


def _extract_og_image(html: str) -> str | None:
    # <meta property="og:image" content="...">
    m = re.search(r"<meta\s+property=\"og:image\"\s+content=\"([^\"]+)\"", html, flags=re.IGNORECASE)
    if not m:
        m = re.search(r"<meta\s+content=\"([^\"]+)\"\s+property=\"og:image\"", html, flags=re.IGNORECASE)
    return m.group(1) if m else None


def _download_file(url: str, out_path: Path) -> None:
    req = urllib.request.Request(url, headers={"User-Agent": UA, "Referer": "https://www.flickr.com/"})
    with urllib.request.urlopen(req, timeout=TIMEOUT_S) as resp:
        data = resp.read()
    out_path.write_bytes(data)


def _ext_from_url(url: str) -> str:
    try:
        p = urllib.parse.urlparse(url)
        ext = Path(p.path).suffix.lower()
        if ext in (".jpg", ".jpeg", ".png", ".webp"):
            return ext
    except Exception:
        pass
    return ".jpg"


local_paths = []
status = []

for i, row in sample.iterrows():
    page_url = str(row[URL_COL])
    try:
        html = _fetch_html(page_url)
        img_url = _extract_og_image(html)
        if not img_url:
            raise RuntimeError("og:image not found on page")

        ext = _ext_from_url(img_url)
        # Use row index + a short slug so filenames are stable
        slug = _safe_filename(page_url.rstrip("/").split("/")[-1] or f"row_{i}")
        fname = f"tokyo_{i:05d}_{slug}{ext}"
        out_path = TOKYO_IMG_DIR / fname

        if not out_path.exists() or out_path.stat().st_size == 0:
            _download_file(img_url, out_path)
            time.sleep(SLEEP_S)

        local_paths.append(str(out_path))
        status.append("ok")

    except Exception as e:
        local_paths.append("")
        status.append(f"fail: {type(e).__name__}: {e}")

sample["local_image_path"] = local_paths
sample["download_status"] = status

ok_n = int((sample["download_status"] == "ok").sum())
fail_n = int((sample["download_status"] != "ok").sum())
print(f"\nDownload results: ok={ok_n}, failed={fail_n}")

# Create output CSV: keep same rows/columns, but replace URL column with local path
# (drop the original URL column and rename local_image_path -> photo/video_page_url)
out_df = sample.drop(columns=[URL_COL]).rename(columns={"local_image_path": URL_COL})

# Optional: keep download_status so you can filter failures later
out_df.to_csv(TOKYO_OUT_CSV, index=False)
print("Wrote:", str(TOKYO_OUT_CSV))

# Quick preview
out_df.head()

tokyo_clean.csv shape: (10000, 7)
columns: ['user_id', 'longitude', 'latitude', 'date_taken', 'photo/video_page_url', 'x', 'y']
Processing rows: 200

Download results: ok=159, failed=41
Wrote: C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\tokyo_first100_local.csv


,user_id,longitude,latitude,date_taken,x,y,photo/video_page_url,download_status
0,10727420@N00,139.700499,35.674000,2010-04-09 17:26:25.0,1.555139e+07,4.255856e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
1,8819274@N04,139.766521,35.709095,2007-02-10 16:08:40.0,1.555874e+07,4.260667e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
2,62068690@N00,139.765632,35.694482,2008-12-21 15:45:31.0,1.555864e+07,4.258664e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
3,49503094041@N01,139.784391,35.548589,2011-11-11 05:48:54.0,1.556073e+07,4.238684e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
4,40443199@N00,139.768753,35.671521,2006-04-06 16:42:49.0,1.555899e+07,4.255517e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok


In [3]:
# ── Filter out failed image downloads from tokyo_first100_local.csv ───────────

from pathlib import Path

TOKYO_LOCAL_CSV = Path(r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/tokyo_first100_local.csv")
TOKYO_FILTERED_CSV = Path(r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/tokyo_first100_local_ok.csv")

URL_COL = "photo/video_page_url"  # in the rewritten CSV this now holds the local file path
STATUS_COL = "download_status"

_df = pd.read_csv(TOKYO_LOCAL_CSV)
print("Loaded:", str(TOKYO_LOCAL_CSV))
print("Shape:", _df.shape)
print("Columns:", list(_df.columns))

if STATUS_COL not in _df.columns:
    raise KeyError(f"'{STATUS_COL}' column not found; can't filter failures")
if URL_COL not in _df.columns:
    raise KeyError(f"'{URL_COL}' column not found; expected local path column")

# Keep rows marked ok AND with an existing local file path
paths = _df[URL_COL].astype(str).str.strip()
ok_mask = _df[STATUS_COL].astype(str).str.strip().eq("ok")
exists_mask = paths.apply(lambda p: Path(p).exists() if p and p.lower() != "nan" else False)

filtered = _df[ok_mask & exists_mask].copy()

print("\nCounts:")
print("- total:", len(_df))
print("- ok by status:", int(ok_mask.sum()))
print("- ok+file_exists:", len(filtered))
print("- removed:", len(_df) - len(filtered))

filtered.to_csv(TOKYO_FILTERED_CSV, index=False)
print("\nWrote filtered CSV:", str(TOKYO_FILTERED_CSV))

filtered.head()

Loaded: C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\tokyo_first100_local.csv
Shape: (200, 8)
Columns: ['user_id', 'longitude', 'latitude', 'date_taken', 'x', 'y', 'photo/video_page_url', 'download_status']

Counts:
- total: 200
- ok by status: 159
- ok+file_exists: 159
- removed: 41

Wrote filtered CSV: C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\tokyo_first100_local_ok.csv


,user_id,longitude,latitude,date_taken,x,y,photo/video_page_url,download_status
0,10727420@N00,139.700499,35.674000,2010-04-09 17:26:25.0,1.555139e+07,4.255856e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
1,8819274@N04,139.766521,35.709095,2007-02-10 16:08:40.0,1.555874e+07,4.260667e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
2,62068690@N00,139.765632,35.694482,2008-12-21 15:45:31.0,1.555864e+07,4.258664e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
3,49503094041@N01,139.784391,35.548589,2011-11-11 05:48:54.0,1.556073e+07,4.238684e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok
4,40443199@N00,139.768753,35.671521,2006-04-06 16:42:49.0,1.555899e+07,4.255517e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,ok


In [4]:
# ── Add captions + remove download_status from the OK CSV ─────────────────────

import random
from pathlib import Path

TOKYO_OK_CSV = Path(r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/tokyo_first100_local_ok.csv")

# Load the "ok" CSV we created earlier
cap_df = pd.read_csv(TOKYO_OK_CSV)

# Remove the status column (it will just be 'ok')
cap_df = cap_df.drop(columns=["download_status"], errors="ignore")

# Random caption pool with varied emotions
CAPTIONS = [
    "Feeling happy and light today.",
    "I’m excited about what comes next.",
    "A calm moment that made me smile.",
    "Today felt peaceful and steady.",
    "I’m proud of myself for showing up.",
    "A little anxious, but still hopeful.",
    "Feeling overwhelmed and tired.",
    "A quiet day, kind of neutral.",
    "Feeling sad and missing home.",
    "I’m stressed, but trying to breathe through it.",
    "Mood is low, but I’m hanging on.",
    "Feeling depressed and disconnected right now.",
    "I’m grateful, even if it’s complicated.",
    "A burst of energy—feeling motivated.",
    "Frustrated today, but I’ll reset tomorrow.",
    "Lonely, but trying to stay grounded.",
    "Hopeful after a small win.",
    "Feeling nostalgic and a bit emotional.",
    "Confident—like I can handle things.",
    "Drained, but proud I made it through.",
]

random.seed(42)
cap_df["caption"] = [random.choice(CAPTIONS) for _ in range(len(cap_df))]

# Overwrite the same CSV (as requested)
cap_df.to_csv(TOKYO_OK_CSV, index=False)
print("Updated CSV written (caption added, download_status removed):", str(TOKYO_OK_CSV))

cap_df.head(10)

Updated CSV written (caption added, download_status removed): C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\tokyo_first100_local_ok.csv


,user_id,longitude,latitude,date_taken,x,y,photo/video_page_url,caption
0,10727420@N00,139.700499,35.674000,2010-04-09 17:26:25.0,1.555139e+07,4.255856e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Today felt peaceful and steady.
1,8819274@N04,139.766521,35.709095,2007-02-10 16:08:40.0,1.555874e+07,4.260667e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Feeling happy and light today.
2,62068690@N00,139.765632,35.694482,2008-12-21 15:45:31.0,1.555864e+07,4.258664e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Feeling sad and missing home.
3,49503094041@N01,139.784391,35.548589,2011-11-11 05:48:54.0,1.556073e+07,4.238684e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,"A quiet day, kind of neutral."
4,40443199@N00,139.768753,35.671521,2006-04-06 16:42:49.0,1.555899e+07,4.255517e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,"A quiet day, kind of neutral."
5,52465044@N00,139.647388,35.548454,2009-01-20 20:33:36.0,1.554548e+07,4.238666e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,I’m proud of myself for showing up.
6,53869907@N00,139.774675,35.626971,2006-10-01 16:12:01.0,1.555965e+07,4.249414e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Today felt peaceful and steady.
7,39569656@N07,139.765942,35.635743,2012-08-10 04:06:16.0,1.555867e+07,4.250615e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Feeling nostalgic and a bit emotional.
8,34106794@N00,139.700517,35.659737,2005-02-06 02:57:36.0,1.555139e+07,4.253902e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,A calm moment that made me smile.
9,57102247@N05,139.615269,35.611735,2012-08-18 20:02:16.0,1.554190e+07,4.247327e+06,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...,Confident—like I can handle things.


In [5]:
# ── Add sequential (irregular) timestamps in 2025 ─────────────────────────────

import random
from datetime import datetime, timedelta
from pathlib import Path

TOKYO_OK_CSV = Path(r"C:/Users/ANISH/OneDrive/Desktop/osC/DREAMS/Other/Dataset/tokyo_first100_local_ok.csv")

df_time = pd.read_csv(TOKYO_OK_CSV)

START = datetime(2025, 1, 1, 0, 0, 0)
END   = datetime(2025, 12, 31, 23, 59, 59)

random.seed(42)

n = len(df_time)
if n == 0:
    raise ValueError("CSV has 0 rows")

# Build irregular day-gaps: many small gaps, some medium, few large.
# This keeps times in sequence but not equally spaced.
def sample_gap_days() -> int:
    r = random.random()
    if r < 0.55:
        return random.randint(0, 2)      # same/next couple days (common)
    if r < 0.85:
        return random.randint(3, 7)      # within a week (sometimes)
    if r < 0.97:
        return random.randint(8, 20)     # a few weeks (rare)
    return random.randint(21, 45)        # longer gaps (very rare)

# Generate cumulative offsets
offset_days = [0]
for _ in range(n - 1):
    offset_days.append(offset_days[-1] + sample_gap_days())

# If we overshoot the year range, compress offsets so the last date fits within END
max_off = offset_days[-1]
max_span_days = (END - START).days
if max_off > max_span_days and max_off > 0:
    scale = max_span_days / max_off
    offset_days = [int(round(d * scale)) for d in offset_days]
    # ensure monotonic non-decreasing after rounding
    for i in range(1, len(offset_days)):
        if offset_days[i] < offset_days[i - 1]:
            offset_days[i] = offset_days[i - 1]

# Create timestamps with random time-of-day, while keeping non-decreasing order
stamps = []
prev = START
for d in offset_days:
    base = START + timedelta(days=int(d))
    # pick a time-of-day
    hh = random.randint(6, 22)
    mm = random.choice([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
    ss = 0
    ts = base.replace(hour=hh, minute=mm, second=ss)
    if ts < prev:
        ts = prev
    if ts > END:
        ts = END
    stamps.append(ts)
    prev = ts

# Assign as ISO timestamp string
# Format requested example: 2026-01-07T11:48:00
# We'll use: YYYY-MM-DDTHH:MM:SS

df_time["Time"] = [t.strftime("%Y-%m-%dT%H:%M:%S") for t in stamps]

# Overwrite the same CSV
df_time.to_csv(TOKYO_OK_CSV, index=False)
print("Updated CSV written (Time added):", str(TOKYO_OK_CSV))

# Validation
parsed = pd.to_datetime(df_time["Time"], errors="coerce")
print("\nValidation:")
print("- rows:", len(df_time))
print("- min Time:", parsed.min())
print("- max Time:", parsed.max())
print("- monotonic non-decreasing:", bool(parsed.is_monotonic_increasing))

# preview
(df_time[["Time", "caption", "photo/video_page_url"]].head(10) if all(c in df_time.columns for c in ["Time","caption","photo/video_page_url"]) else df_time.head(10))

Updated CSV written (Time added): C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Other\Dataset\tokyo_first100_local_ok.csv

Validation:
- rows: 159
- min Time: 2025-01-01 22:00:00
- max Time: 2025-12-31 13:15:00
- monotonic non-decreasing: True


,Time,caption,photo/video_page_url
0,2025-01-01T22:00:00,Today felt peaceful and steady.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
1,2025-01-02T15:50:00,Feeling happy and light today.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
2,2025-01-04T09:10:00,Feeling sad and missing home.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
3,2025-01-05T14:05:00,"A quiet day, kind of neutral.",C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
4,2025-01-06T09:55:00,"A quiet day, kind of neutral.",C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
5,2025-01-11T10:20:00,I’m proud of myself for showing up.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
6,2025-01-12T15:45:00,Today felt peaceful and steady.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
7,2025-01-12T15:45:00,Feeling nostalgic and a bit emotional.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
8,2025-01-13T16:15:00,A calm moment that made me smile.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
9,2025-01-13T16:15:00,Confident—like I can handle things.,C:\Users\ANISH\OneDrive\Desktop\osC\DREAMS\Oth...
